# GPU Preprocessing Runner (Local VS Code Notebook)

This notebook runs the project CLI scripts on a local machine with GPU support.

What you need to edit:
- `REPO_DIR` (local repository path)
- `INPUT_DIR` (folder with your 2D slices)

Pipeline executed by the main script:
1. Stack slices into a 3D volume
2. Apply norm200 normalization
3. Run CUDA NLM (chunked)
4. Save outputs into `norm200_output/` and `nlm_output/`

## 1) Local runtime setup
Use the VS Code Jupyter kernel from `venv-napari` for local GPU execution.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

print('Python:', sys.version)
print('Python executable:', sys.executable)
print('Working dir:', os.getcwd())

In [ ]:
# Local GPU check (PyTorch-based)
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU count:', torch.cuda.device_count())
    print('GPU name:', torch.cuda.get_device_name(0))

In [ ]:
# Optional local dependency check (no install command needed)
required = ['numpy', 'tifffile', 'torch']
for pkg in required:
    try:
        __import__(pkg)
        print(f'OK: {pkg}')
    except Exception as e:
        print(f'MISSING: {pkg} -> {e}')

## 2) Confirm paths
Defaults below are set for your local workspace, input slices folder, and export target folder.

In [ ]:
REPO_DIR = Path(r'C:\Users\rony.schwartz\Documents\nnUNet4SoilXrayCT')
INPUT_DIR = Path(r'\\HIVE3065\Yael_Mishael\Rony\10.12.25_Rehovot_samp_2\Rehovot_samp2_highkV_Cu0.11_15um_Rec')
EXPORT_DIR = Path(r'\\HIVE3065\Yael_Mishael\Rony\remote_computer backup\10.5')
EXPORT_BASENAME = 'rehovot_samp_2'

print('REPO_DIR =', REPO_DIR)
print('INPUT_DIR =', INPUT_DIR)
print('EXPORT_DIR =', EXPORT_DIR)
print('EXPORT_BASENAME =', EXPORT_BASENAME)

if not REPO_DIR.exists():
    raise FileNotFoundError(f'Repository not found: {REPO_DIR}')
if not INPUT_DIR.exists():
    raise FileNotFoundError(f'Input folder not found: {INPUT_DIR}')
if not EXPORT_DIR.exists():
    raise FileNotFoundError(f'Export folder not found: {EXPORT_DIR}')

## 3) Quick CLI check
This verifies the script is callable and shows CLI help using the current kernel interpreter.

In [ ]:
cmd = [sys.executable, 'preprocess/run_preprocess.py', '--help']
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=str(REPO_DIR), check=True)

## 4) Run the main GPU preprocessing CLI
This executes stack -> norm200 -> CUDA NLM (chunked).

In [ ]:
cmd = [
    sys.executable,
    'preprocess/run_preprocess.py',
    '--input_dir',
    str(INPUT_DIR),
]

print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=str(REPO_DIR), check=True)

## 5) Validate outputs and export final TIF
Expected intermediate outputs:
- `preprocess/norm200_output/norm200_volume.tif`
- `preprocess/nlm_output/nlm_volume.tif`

Final export target:
- `\\HIVE3065\Yael_Mishael\Rony\remote_computer backup\10.5\rehovot_samp_2.tif`

In [ ]:
import shutil
import tifffile

norm_path = REPO_DIR / 'preprocess' / 'norm200_output' / 'norm200_volume.tif'
nlm_path = REPO_DIR / 'preprocess' / 'nlm_output' / 'nlm_volume.tif'
export_path = EXPORT_DIR / f'{EXPORT_BASENAME}.tif'

print('norm path exists:', norm_path.exists(), norm_path)
print('nlm path exists:', nlm_path.exists(), nlm_path)
print('export path:', export_path)

if norm_path.exists():
    norm_vol = tifffile.imread(norm_path)
    print('norm200 shape:', norm_vol.shape, 'dtype:', norm_vol.dtype)

if nlm_path.exists():
    nlm_vol = tifffile.imread(nlm_path)
    print('nlm shape:', nlm_vol.shape, 'dtype:', nlm_vol.dtype)

    shutil.copy2(nlm_path, export_path)
    print('Exported final TIF to:', export_path)
else:
    raise FileNotFoundError(f'Expected NLM output not found: {nlm_path}')

## Optional: run the playground CLI (desktop workflow)
This command is optional and can be used on a local machine with a GUI setup.

In [ ]:
# Example only (optional):
# subprocess.run([
#     sys.executable, 'preprocess_playground/run_napari_filters.py',
#     '--input_dir', str(INPUT_DIR),
#     '--filter', 'nlm'
# ], cwd=str(REPO_DIR), check=True)